# Food-101 classification baseline

Establishes the number every later stage is measured against.

The notebook is deliberately thin. All the logic lives in the `platevision` package in the
repository, which is unit tested in CI. Notebooks are a bad place to keep code you intend to
trust: they cannot be linted meaningfully, they cannot be tested, and they encourage editing
the thing you are trying to measure.

**Before running:** Settings, Accelerator, GPU T4 x2 (or P100), and turn Internet on.

Expect roughly 2 to 4 hours for 30 epochs on a T4.

## 1. Install the package

In [ ]:
!git clone --depth 1 https://github.com/simonkundrik/plate-vision.git /kaggle/working/plate-vision
%pip install -q -e /kaggle/working/plate-vision/model[train,export]

import torch

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 2. Fetch Food-101

Uses the repository's own downloader rather than a mounted Kaggle dataset, because it
verifies the class ordering against `shared/food101_labels.json`. Index N in that file is
position N on the logits axis of the exported model, and the canonical Food-101 order is not
alphabetical (`cheesecake` precedes `cheese_plate`). A community mirror with a re-sorted
`classes.txt` would train perfectly and mislabel every prediction.

Roughly 5 GB down, 5 GB extracted. The archive is deleted after extraction.

In [ ]:
%cd /kaggle/working/plate-vision/model
!python data/download_food101.py --out /kaggle/temp/food101

## 3. Smoke run

Two epochs on five classes with a small backbone. This is here to catch a broken data path
or a mis-specified transform in two minutes rather than two hours.

In [ ]:
!python scripts/train_classifier.py \
    --data-root /kaggle/temp/food101/food-101 \
    --out /kaggle/working/runs/smoke \
    --backbone mobilenetv3_small_100 \
    --subset-classes 5 --limit-train 500 --limit-val 200 \
    --epochs 2 --batch-size 32 --workers 2 --amp

## 4. Baseline run

EfficientNet-B0, all 101 classes, plain cross-entropy. No mixup, no cutmix, no EMA, no label
smoothing. Those land in the next PR specifically so their contribution can be measured
against this run instead of assumed.

In [ ]:
!python scripts/train_classifier.py \
    --data-root /kaggle/temp/food101/food-101 \
    --out /kaggle/working/runs/baseline \
    --backbone efficientnet_b0 \
    --epochs 30 --batch-size 128 --lr 1e-3 --weight-decay 0.05 \
    --workers 4 --amp

## 5. Curves

Save `history.json` and `best.pt` from the output pane. `history.json` is what the ablation
table in the next PR is built from.

In [ ]:
import json

import matplotlib.pyplot as plt

history = json.loads(open("/kaggle/working/runs/baseline/history.json").read())
train = [e for e in history if e["split"] == "train"]
val = [e for e in history if e["split"] == "val"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot([e["epoch"] for e in train], [e["loss"] for e in train], label="train")
ax1.plot([e["epoch"] for e in val], [e["loss"] for e in val], label="val")
ax1.set_xlabel("epoch")
ax1.set_ylabel("loss")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot([e["epoch"] for e in train], [e["top1"] for e in train], label="train")
ax2.plot([e["epoch"] for e in val], [e["top1"] for e in val], label="val")
ax2.set_xlabel("epoch")
ax2.set_ylabel("top-1 %")
ax2.legend()
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best = max(val, key=lambda e: e["top1"])
print(f"best val top-1: {best['top1']:.2f}% at epoch {best['epoch']}")
print(f"best val top-5: {max(e['top5'] for e in val):.2f}%")